# Week 2 — Flower Classifier & House Price Predictor

**Theme:** Supervised learning I — k-nearest neighbors & linear regression

Supervised learning means: we have labeled examples (input -> correct answer),
and we want the computer to learn the pattern well enough to predict the answer
for *new*, unseen inputs. There are two flavors:

- **Classification** — the answer is a category (e.g. which species of flower)
- **Regression** — the answer is a number (e.g. a price, a score)

We'll do one of each.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris, load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, mean_squared_error, r2_score

## Part A — Classification: k-Nearest Neighbors

**Idea:** to classify a new flower, look at its `k` closest neighbors (by
measurement) among the flowers we already know the species of, and take a vote.

The classic **Iris** dataset: 150 flowers, 4 measurements each (sepal/petal
length & width), 3 species.

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
print("Features:", iris.feature_names)
print("Species:", iris.target_names)
print("Shape:", X.shape)

In [ ]:
# Split into training data (learn from) and test data (evaluate on, never trained on)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

predictions = knn.predict(X_test)
accuracy = accuracy_score(y_test, predictions)
print(f"Test accuracy with k=5: {accuracy:.2%}")

In [ ]:
# Does the choice of k matter? Try a range of values.
accuracies = []
k_values = range(1, 21)
for k in k_values:
    model = KNeighborsClassifier(n_neighbors=k)
    model.fit(X_train, y_train)
    accuracies.append(accuracy_score(y_test, model.predict(X_test)))

plt.figure(figsize=(6, 4))
plt.plot(list(k_values), accuracies, marker="o")
plt.title("k-NN Accuracy vs. k")
plt.xlabel("k (number of neighbors)")
plt.ylabel("Test accuracy")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Visualize the decision boundary using just 2 of the 4 features, so we can plot it
X2 = X[:, 2:4]  # petal length, petal width
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y, test_size=0.3, random_state=42, stratify=y
)
knn2 = KNeighborsClassifier(n_neighbors=5).fit(X2_train, y2_train)

xx, yy = np.meshgrid(
    np.linspace(X2[:, 0].min() - 0.5, X2[:, 0].max() + 0.5, 200),
    np.linspace(X2[:, 1].min() - 0.5, X2[:, 1].max() + 0.5, 200),
)
Z = knn2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

plt.figure(figsize=(6, 5))
plt.contourf(xx, yy, Z, alpha=0.25, cmap="viridis")
plt.scatter(X2[:, 0], X2[:, 1], c=y, cmap="viridis", edgecolor="k")
plt.title("k-NN Decision Boundary (petal length vs. width)")
plt.xlabel(iris.feature_names[2])
plt.ylabel(iris.feature_names[3])
plt.show()

## Part B — Regression: Linear Regression

**Idea:** fit a straight line (or plane, in higher dimensions) through the data
that best predicts a numeric target.

The **diabetes** dataset: 442 patients, several health measurements, and a
target that measures disease progression one year later. We'll start with just
one feature — BMI — so we can plot the fitted line directly.

In [ ]:
diabetes = load_diabetes()
bmi = diabetes.data[:, diabetes.feature_names.index("bmi")].reshape(-1, 1)
target = diabetes.target

Xb_train, Xb_test, yb_train, yb_test = train_test_split(
    bmi, target, test_size=0.3, random_state=42
)

reg = LinearRegression()
reg.fit(Xb_train, yb_train)

print(f"Learned line: progression = {reg.coef_[0]:.1f} * bmi + {reg.intercept_:.1f}")

In [ ]:
predictions = reg.predict(Xb_test)
print(f"R^2 score:  {r2_score(yb_test, predictions):.3f}  (1.0 = perfect, 0.0 = no better than guessing the mean)")
print(f"RMSE:       {mean_squared_error(yb_test, predictions) ** 0.5:.1f}")

In [ ]:
plt.figure(figsize=(6, 5))
plt.scatter(Xb_test, yb_test, alpha=0.6, label="actual")
order = np.argsort(Xb_test[:, 0])
plt.plot(Xb_test[order], predictions[order], color="red", linewidth=2, label="predicted line")
plt.title("Linear Regression: BMI -> Disease Progression")
plt.xlabel("BMI (standardized)")
plt.ylabel("Disease progression score")
plt.legend()
plt.show()

## Try it yourself

1. **Pick your own k.** Looking at the accuracy-vs-k plot above, which `k`
   would you choose, and why might a very small or very large `k` be a bad idea?
2. **Classify a new flower by hand.** Make up 4 measurements
   (`knn.predict([[5.1, 3.5, 1.4, 0.2]])`) and see which species k-NN predicts.
3. **Add a second feature to the regression.** Use both `bmi` and `s5` (another
   column in `diabetes.feature_names`) as inputs to `LinearRegression` — does
   the R² improve?
4. **Compare to a "dumb" baseline.** What R² would you get if you always
   predicted the *average* disease progression, regardless of BMI? (Hint:
   that's what an R² of 0 means.)

---
## 🎯 캡스톤: 이번 학기 성적 위험도 예측기

가상의 선배 150명의 "주당 공부시간 / 출석률 / 평균 수면시간 -> 기말 점수" 기록을 드립니다. 이 데이터로 **k-NN 분류기**(위험군 Safe/Warning/Danger 예측)와 **선형회귀**(예상 점수 예측)를 직접 만들어보고, 마지막엔 **여러분 자신의 예상 습관**을 입력해서 결과를 확인해보세요.

**확장 아이디어:** 학기 말에 실제 본인의 공부시간/출석/수면 기록과 실제 성적을 몇 학기치 모아서 `students_df`를 바꿔치기하면, 진짜 "내 성적 예측기"가 됩니다.

In [ ]:
# 더미 데이터 생성 (실행만 하면 됩니다)
import pandas as pd
rng = np.random.default_rng(7)
n_students = 150

weekly_study_hours = np.clip(rng.normal(10, 4, n_students), 0, 25)
attendance_rate = np.clip(rng.normal(0.85, 0.12, n_students), 0.4, 1.0)
sleep_hours_avg = np.clip(rng.normal(6.5, 1.2, n_students), 3, 10)

final_score = (
    20
    + 2.2 * weekly_study_hours
    + 45 * attendance_rate
    + 1.5 * sleep_hours_avg
    + rng.normal(0, 6, n_students)
)
final_score = np.clip(final_score, 0, 100)

def to_risk(score):
    if score >= 80:
        return "Safe"
    elif score >= 60:
        return "Warning"
    return "Danger"

students_df = pd.DataFrame({
    "weekly_study_hours": weekly_study_hours.round(1),
    "attendance_rate": attendance_rate.round(2),
    "sleep_hours_avg": sleep_hours_avg.round(1),
    "final_score": final_score.round(1),
})
students_df["risk"] = students_df["final_score"].apply(to_risk)
students_df.head()

### 여러분의 과제

1. `students_df`에서 `weekly_study_hours`, `attendance_rate`, `sleep_hours_avg` 3개를 입력(X)으로, `risk`를 정답(y)으로 하여 **k-NN 분류기**를 학습시키고 테스트 정확도를 출력하세요. (Part A 코드를 참고하세요: `train_test_split`, `KNeighborsClassifier`, `accuracy_score`)
2. 같은 3개 입력으로 `final_score`(숫자)를 예측하는 **선형회귀 모델**을 학습시키고 R² 점수를 출력하세요. (Part B 코드 참고: `LinearRegression`, `r2_score`)
3. 아래에 **여러분 자신의 예상 습관**(예상 주당 공부시간, 예상 출석률, 예상 평균 수면시간)을 숫자로 입력하고, 학습된 두 모델로 (a) 위험군과 (b) 예상 점수를 각각 예측해서 출력해보세요.

In [ ]:
# TODO 1: k-NN 분류기로 risk(Safe/Warning/Danger)를 예측하는 모델을 학습하고 테스트 정확도를 출력하세요.


# TODO 2: 선형회귀로 final_score를 예측하는 모델을 학습하고 R^2를 출력하세요.


# TODO 3: 나의 예상 습관을 입력하고, 위 두 모델로 위험군과 예상 점수를 예측해보세요.
my_weekly_study_hours = None   # 예: 8
my_attendance_rate = None      # 예: 0.9
my_sleep_hours_avg = None      # 예: 6.5